In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import MinMaxScaler


In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

## Test dataset: MAASTRO 

In [5]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [6]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [7]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [8]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [9]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [10]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [11]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [12]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [13]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [14]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [15]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [16]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"glcm_ClusterShade_d_1_PET_b2",
"shape_MajorAxisLength",
"gldm_DependenceEntropy_d_1_PET_c04",
"ngtdm_Strength_d_1_CT_b20",
"LBP_120_PET",
"glcm_ClusterProminence_d_1_PET_b2"
] 

In [17]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [18]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [19]:
# Copy the original X for later 
original_X = X.copy()

In [20]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = MinMaxScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [21]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [22]:
X_new

,glcm_ClusterShade_d_1_PET_b2,shape_MajorAxisLength,gldm_DependenceEntropy_d_1_PET_c04,ngtdm_Strength_d_1_CT_b20,LBP_120_PET,glcm_ClusterProminence_d_1_PET_b2
0,10.600998,42.073251,6.795642,4.288423,0.140311,434.830466
1,1.734460,24.613845,6.637753,9.401886,0.191058,19.257701
2,1.761780,48.030294,6.894124,3.973266,0.126531,29.527599
3,0.159909,25.589900,6.578202,8.266749,0.192388,1.515103
4,-0.177915,34.684750,6.800483,4.402019,0.202073,10.006413
...,...,...,...,...,...,...
134,9.578449,33.069705,6.662260,1.177483,0.152626,109.970854
135,-0.557128,41.043692,6.874148,4.537614,0.142778,12.574172
136,28.029294,36.618802,6.659339,3.036865,0.140582,386.172283
137,0.111923,45.870392,6.812758,2.379513,0.156640,15.793118


In [23]:
X_new_std

,glcm_ClusterShade_d_1_PET_b2,shape_MajorAxisLength,gldm_DependenceEntropy_d_1_PET_c04,ngtdm_Strength_d_1_CT_b20,LBP_120_PET,glcm_ClusterProminence_d_1_PET_b2
0,0.248118,0.350904,0.865889,0.098124,0.174091,0.052658
1,0.222288,0.123069,0.788611,0.227868,0.456736,0.002252
2,0.222368,0.428640,0.914091,0.090128,0.097341,0.003498
3,0.217702,0.135806,0.759464,0.199066,0.464144,0.000100
4,0.216717,0.254489,0.868259,0.101007,0.518081,0.001130
...,...,...,...,...,...,...
134,0.245139,0.233413,0.800606,0.019191,0.242682,0.013255
135,0.215613,0.337469,0.904313,0.104447,0.187835,0.001442
136,0.298890,0.279727,0.799176,0.066369,0.175602,0.046756
137,0.217562,0.400455,0.874266,0.049690,0.265036,0.001832


In [24]:
MAASTRO_new 

,glcm_ClusterShade_d_1_PET_b2,shape_MajorAxisLength,gldm_DependenceEntropy_d_1_PET_c04,ngtdm_Strength_d_1_CT_b20,LBP_120_PET,glcm_ClusterProminence_d_1_PET_b2
0,-6.132486,50.002093,6.814563,1.344991,0.122209,479.762323
1,11.365721,41.753334,6.154990,9.021133,0.128976,93.366330
2,18.556645,44.375483,6.673632,4.686304,0.137282,393.917369
3,6.670130,46.115989,6.577609,8.227208,0.171595,81.628434
4,6.613032,54.394967,6.735323,3.416011,0.128134,61.795989
...,...,...,...,...,...,...
94,467.061873,34.218615,6.559987,8.997680,0.137194,19154.348390
95,34.619690,51.046869,6.579052,3.803795,0.137620,368.948026
96,4.698589,50.417953,6.675182,3.847965,0.115099,33.083289
97,20.628059,44.901412,6.636306,2.686505,0.117654,404.363216


In [25]:
MAASTRO_new_std

,glcm_ClusterShade_d_1_PET_b2,shape_MajorAxisLength,gldm_DependenceEntropy_d_1_PET_c04,ngtdm_Strength_d_1_CT_b20,LBP_120_PET,glcm_ClusterProminence_d_1_PET_b2
0,0.199371,0.454371,0.875150,0.023441,0.073271,0.058108
1,0.250346,0.346729,0.552325,0.218207,0.110963,0.011241
2,0.271294,0.380947,0.806172,0.108220,0.157221,0.047695
3,0.236667,0.403660,0.759174,0.198063,0.348334,0.009817
4,0.236501,0.511695,0.836366,0.075989,0.106269,0.007412
...,...,...,...,...,...,...
94,1.577870,0.248406,0.750549,0.217612,0.156731,2.323190
95,0.318089,0.468005,0.759880,0.085828,0.159105,0.044667
96,0.230923,0.459798,0.806930,0.086949,0.033672,0.003929
97,0.277329,0.387810,0.787903,0.057479,0.047900,0.048962


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [26]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 13:53:17,347] A new study created in memory with name: no-name-daccdf6c-ea10-46cf-b7ff-49dfb00b062c


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6877637130801688


[I 2024-04-17 13:53:25,068] A new study created in memory with name: no-name-7d0df791-cd39-4438-a2ff-7f247158aafc


Fold 5 C-index: 0.7370892018779343
[I 2024-04-17 13:53:25,059] Trial 0 finished with value: 0.7221538653954912 and parameters: {}. Best is trial 0 with value: 0.7221538653954912.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7221538653954912], datetime_start=datetime.datetime(2024, 4, 17, 13, 53, 17, 422050), datetime_complete=datetime.datetime(2024, 4, 17, 13, 53, 25, 59293), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7221538653954912


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.18540223318183244
Fold 2 IBS: 0.15964445582496828
Fold 3 IBS: 0.141452621684816
Fold 4 IBS: 0.21107747669358115
Fold 5 IBS: 0.16928388234449362
[I 2024-04-17 13:53:25,674] Trial 0 finished with value: 0.17337213394593828 and parameters: {}. Best is trial 0 with value: 0.17337213394593828.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17337213394593828], datetime_start=datetime.datetime(2024, 4, 17, 13, 53, 25, 127808), datetime_complete=datetime.datetime(2024, 4, 17, 13, 53, 25, 674125), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17337213394593828


In [27]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [28]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.722
train_ibs:  0.173


#### Test

In [29]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [30]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.58
IBS score: 0.246


In [31]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [32]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [33]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:53:25,997] A new study created in memory with name: no-name-9725a503-fa26-4847-95a3-2040c5af6357


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6017316017316018


[I 2024-04-17 13:53:26,253] A new study created in memory with name: no-name-7fec3fdc-2b79-4332-a277-3ee19e1f25c5


Fold 2 C-index: 0.734375
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.6807511737089202
[I 2024-04-17 13:53:26,232] Trial 0 finished with value: 0.6781791996649223 and parameters: {}. Best is trial 0 with value: 0.6781791996649223.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6781791996649223], datetime_start=datetime.datetime(2024, 4, 17, 13, 53, 26, 34621), datetime_complete=datetime.datetime(2024, 4, 17, 13, 53, 26, 231741), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6781791996649223


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397652137310474
Fold 2 IBS: 0.22157791529947765
Fold 3 IBS: 0.2045359462877086
Fold 4 IBS: 0.2247380399747716
Fold 5 IBS: 0.21812431382491174
[I 2024-04-17 13:53:26,478] Trial 0 finished with value: 0.2165905473519949 and parameters: {}. Best is trial 0 with value: 0.2165905473519949.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905473519949], datetime_start=datetime.datetime(2024, 4, 17, 13, 53, 26, 292719), datetime_complete=datetime.datetime(2024, 4, 17, 13, 53, 26, 478062), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905473519949


In [34]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [35]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.678
train_ibs:  0.217


#### Test

In [36]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [37]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.563


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [38]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [39]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:53:26,752] A new study created in memory with name: no-name-35354e6d-f4b9-40cd-93b0-87204ce6ff31


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697


[I 2024-04-17 13:53:27,221] A new study created in memory with name: no-name-ff59cec0-15eb-4f3e-a6a7-84fa9ca9bb87


Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:27,211] Trial 0 finished with value: 0.723941851655977 and parameters: {}. Best is trial 0 with value: 0.723941851655977.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.723941851655977], datetime_start=datetime.datetime(2024, 4, 17, 13, 53, 26, 794990), datetime_complete=datetime.datetime(2024, 4, 17, 13, 53, 27, 211453), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.723941851655977


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.18681207190116883
Fold 2 IBS: 0.15997942375885582
Fold 3 IBS: 0.14332080814949552
Fold 4 IBS: 0.20639363524620555
Fold 5 IBS: 0.16928556717267654
[I 2024-04-17 13:53:27,834] Trial 0 finished with value: 0.17315830124568046 and parameters: {}. Best is trial 0 with value: 0.17315830124568046.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17315830124568046], datetime_start=datetime.datetime(2024, 4, 17, 13, 53, 27, 264542), datetime_complete=datetime.datetime(2024, 4, 17, 13, 53, 27, 833680), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17315830124568046


In [40]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [41]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.724
train_ibs:  0.173


#### Test

In [42]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [43]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.579


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.245


In [44]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [45]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:53:28,278] A new study created in memory with name: no-name-d70d43ea-0d1b-4495-903e-14e73722822b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:28,851] Trial 0 finished with value: 0.723941851655977 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.723941851655977.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:29,420] Trial 1 finished with value: 0.723941851655977 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.723941851655977.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:30,137] Trial 2 finished with value: 0.723941851655977 and parameters: {'l1_ratio': 0.22692876841884668}. Best

Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:41,626] Trial 24 finished with value: 0.723941851655977 and parameters: {'l1_ratio': 0.3109747602700048}. Best is trial 22 with value: 0.7373347087988342.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:41,956] Trial 25 finished with value: 0.7373347087988342 and parameters: {'l1_ratio': 0.1309253064678097}. Best is trial 22 with value: 0.7373347087988342.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 13:53:42,122] Trial 26 finished with value: 0.6818827456832424 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 22 with value: 0

Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:50,613] Trial 48 finished with value: 0.723941851655977 and parameters: {'l1_ratio': 0.908082736959313}. Best is trial 22 with value: 0.7373347087988342.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.6964285714285714
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.7417840375586855
[I 2024-04-17 13:53:51,082] Trial 49 finished with value: 0.723941851655977 and parameters: {'l1_ratio': 0.26037470874224194}. Best is trial 22 with value: 0.7373347087988342.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 13:53:51,178] Trial 50 finished with value: 0.6818827456832424 and parameters: {'l1_ratio': 0.05069122998976218}. Best is trial 22 with value: 0.7373347087988342.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.1868661895573752
Fold 2 IBS: 0.15996581586464073
Fold 3 IBS: 0.14278778872716694
Fold 4 IBS: 0.20633899833847044
Fold 5 IBS: 0.16915346219645436
[I 2024-04-17 13:54:12,493] Trial 0 finished with value: 0.17302245093682153 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.17302245093682153.
Fold 1 IBS: 0.18676120750172592
Fold 2 IBS: 0.1599465668277593
Fold 3 IBS: 0.14259238633160734
Fold 4 IBS: 0.2064408820565228
Fold 5 IBS: 0.16905998197764482
[I 2024-04-17 13:54:13,308] Trial 1 finished with value: 0.172960204939052 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.172960204939052.
Fold 1 IBS: 0.1867403860040557
Fold 2 IBS: 0.15994744082172474
Fold 3 IBS: 0.1425754033421998
Fold 4 IBS: 0.20637932090081745
Fold 5 IBS: 0.16905210993741496
[I 2024-04-17 13:54:14,202] Trial 2 finished with value: 0.17293893220124254 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.17293893220124254.

Fold 2 IBS: 0.1599533681552424
Fold 3 IBS: 0.14259090976811764
Fold 4 IBS: 0.20659528870793864
Fold 5 IBS: 0.16904567727688483
[I 2024-04-17 13:54:25,433] Trial 25 finished with value: 0.17298881506505936 and parameters: {'l1_ratio': 0.20598575943102115}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.1867621854676154
Fold 2 IBS: 0.21905770902812544
Fold 3 IBS: 0.1424912290744978
Fold 4 IBS: 0.2228340751658362
Fold 5 IBS: 0.16902994874028374
[I 2024-04-17 13:54:25,844] Trial 26 finished with value: 0.18803502949527173 and parameters: {'l1_ratio': 0.12316751626230191}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.1868026995869248
Fold 2 IBS: 0.15994854915214712
Fold 3 IBS: 0.14272592027891043
Fold 4 IBS: 0.20632188062704743
Fold 5 IBS: 0.16910368615510618
[I 2024-04-17 13:54:26,289] Trial 27 finished with value: 0.17298054716002717 and parameters: {'l1_ratio': 0.5534318442311428}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.1869146

Fold 1 IBS: 0.1866852379146352
Fold 2 IBS: 0.15996364584054212
Fold 3 IBS: 0.1425282230084562
Fold 4 IBS: 0.20650615759914515
Fold 5 IBS: 0.16903803143148782
[I 2024-04-17 13:54:39,023] Trial 51 finished with value: 0.1729442591588533 and parameters: {'l1_ratio': 0.19981378826499338}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.18671846261344824
Fold 2 IBS: 0.15995769465172532
Fold 3 IBS: 0.142658428921506
Fold 4 IBS: 0.20633429419471444
Fold 5 IBS: 0.16906120645256603
[I 2024-04-17 13:54:39,502] Trial 52 finished with value: 0.172946017366792 and parameters: {'l1_ratio': 0.3452970797066381}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.18671705987357146
Fold 2 IBS: 0.15995096644959964
Fold 3 IBS: 0.14265643907324752
Fold 4 IBS: 0.20647937397834248
Fold 5 IBS: 0.16907415897564054
[I 2024-04-17 13:54:40,081] Trial 53 finished with value: 0.1729755996700803 and parameters: {'l1_ratio': 0.4031499425622821}. Best is trial 18 with value: 0.172928680724

Fold 5 IBS: 0.1690668050271532
[I 2024-04-17 13:54:50,739] Trial 75 finished with value: 0.17297716271652047 and parameters: {'l1_ratio': 0.3322062680413627}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.1867408748601879
Fold 2 IBS: 0.15994581208577285
Fold 3 IBS: 0.14257549749383414
Fold 4 IBS: 0.20641655143511087
Fold 5 IBS: 0.16905551947916622
[I 2024-04-17 13:54:51,336] Trial 76 finished with value: 0.17294685107081437 and parameters: {'l1_ratio': 0.24646821685745576}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.18681510818488462
Fold 2 IBS: 0.15994983711626284
Fold 3 IBS: 0.14264073138627673
Fold 4 IBS: 0.2064729778773398
Fold 5 IBS: 0.16907082628621445
[I 2024-04-17 13:54:51,845] Trial 77 finished with value: 0.17298989617019567 and parameters: {'l1_ratio': 0.37780650608149935}. Best is trial 18 with value: 0.17292868072423867.
Fold 1 IBS: 0.18678429972013402
Fold 2 IBS: 0.1599593596060318
Fold 3 IBS: 0.14251390432650685
Fold 4 IBS: 0.206513

In [46]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [47]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.737
train_ibs:  0.173


#### Test

In [48]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [49]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.13361135845786085)

test_cindex : 0.579


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.19281755699206077)

test_ibs:  0.244


In [50]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [51]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 13:55:03,224] A new study created in memory with name: no-name-a87dbcff-274c-4777-b76b-f82c305dc85d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5844155844155844
Fold 2 C-index: 0.71875
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.7194092827004219
Fold 5 C-index: 0.7323943661971831
[I 2024-04-17 13:55:07,700] Trial 0 finished with value: 0.6931507094077359 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6931507094077359.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.6470588235294118
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7511737089201878
[I 2024-04-17 13:55:10,594] Trial 1 finished with value: 0.664770691376763 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_feature

Fold 5 C-index: 0.6807511737089202
[I 2024-04-17 13:55:52,477] Trial 16 finished with value: 0.71335747695973 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 20, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.8484814588885312, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08082385841318356, 'warm_start': True}. Best is trial 14 with value: 0.7670453047262116.
Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.7489451476793249
Fold 5 C-index: 0.7699530516431925
[I 2024-04-17 13:55:53,423] Trial 17 finished with value: 0.7416632023804188 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 15, 'max_depth': 17, 'n_estimators': 106, 'oob_score': True, 'max_samples': 0.9817072304699028, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.16544550098450153, 'warm_start': True}. Best is trial 14 with value: 0.767045304726

Fold 1 C-index: 0.5757575757575758
Fold 2 C-index: 0.9196428571428571
Fold 3 C-index: 0.9019607843137255
Fold 4 C-index: 0.9113924050632911
Fold 5 C-index: 0.9530516431924883
[I 2024-04-17 13:56:11,226] Trial 31 finished with value: 0.8523610530939875 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 229, 'oob_score': True, 'max_samples': 0.6917950833213685, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.014219617355821348, 'warm_start': True}. Best is trial 31 with value: 0.8523610530939875.
Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.9151785714285714
Fold 3 C-index: 0.9019607843137255
Fold 4 C-index: 0.9240506329113924
Fold 5 C-index: 0.9577464788732394
[I 2024-04-17 13:56:13,283] Trial 32 finished with value: 0.8558046095227017 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 1, 'max_depth': 13, 'n_estimators': 230, 'oob_score': True, 'max_samples': 0.69890796039871

...

Fold 1 IBS: 0.2137371773904688
Fold 2 IBS: 0.14754581156917576
Fold 3 IBS: 0.19507773705994372
Fold 4 IBS: 0.163288198777688
Fold 5 IBS: 0.16776329561114334
[I 2024-04-17 14:01:00,788] Trial 32 finished with value: 0.17748244408168393 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 4, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 414, 'oob_score': True, 'max_samples': 0.7181306651480585, 'max_features': None, 'min_weight_fraction_leaf': 0.13587869431214727}. Best is trial 13 with value: 0.17616034433425964.
Fold 1 IBS: 0.19767026532023416
Fold 2 IBS: 0.19261585838585468
Fold 3 IBS: 0.1783911196474173
Fold 4 IBS: 0.16382095692225104
Fold 5 IBS: 0.18359776371304465
[I 2024-04-17 14:01:08,233] Trial 33 finished with value: 0.18321919279776036 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 3, 'min_samples_leaf': 12, 'max_depth': 3, 'n_estimators': 414, 'oob_score': True, 'max_samples': 0.6267475337287517, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.21398467576624647
Fold 2 IBS: 0.22137341375753541
Fold 3 IBS: 0.204866353457904
Fold 4 IBS: 0.2246611229640241
Fold 5 IBS: 0.21854516772012514
[I 2024-04-17 14:02:47,524] Trial 48 finished with value: 0.21668614673316702 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 20, 'n_estimators': 313, 'oob_score': False, 'max_samples': 0.7858952696573706, 'max_features': None, 'min_weight_fraction_leaf': 0.39308816597832114}. Best is trial 13 with value: 0.17616034433425964.
Fold 1 IBS: 0.21227701143752029
Fold 2 IBS: 0.19801412173527871
Fold 3 IBS: 0.19081561304604946
Fold 4 IBS: 0.19186135351143677
Fold 5 IBS: 0.19928102321132793
[I 2024-04-17 14:02:50,572] Trial 49 finished with value: 0.1984498245883226 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 13, 'min_samples_leaf': 20, 'max_depth': 17, 'n_estimators': 187, 'oob_score': False, 'max_samples': 0.8459565404966409, 'max_features': 'auto', 'min_weight_fraction_le

Fold 1 IBS: 0.20061974377556896
Fold 2 IBS: 0.19195992762207684
Fold 3 IBS: 0.18557737837059887
Fold 4 IBS: 0.1615581866550213
Fold 5 IBS: 0.1802485594679016
[I 2024-04-17 14:05:33,709] Trial 64 finished with value: 0.18399275917823352 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 3, 'min_samples_leaf': 11, 'max_depth': 10, 'n_estimators': 291, 'oob_score': True, 'max_samples': 0.8450605263280312, 'max_features': None, 'min_weight_fraction_leaf': 0.22089138895350868}. Best is trial 13 with value: 0.17616034433425964.
Fold 1 IBS: 0.2193229187697933
Fold 2 IBS: 0.1890601235291585
Fold 3 IBS: 0.17894888493273703
Fold 4 IBS: 0.1832473653779553
Fold 5 IBS: 0.18959401711872684
[I 2024-04-17 14:05:45,468] Trial 65 finished with value: 0.19203466194567417 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 4, 'min_samples_leaf': 12, 'max_depth': 11, 'n_estimators': 459, 'oob_score': True, 'max_samples': 0.79560619712759, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.1988695369031691
Fold 2 IBS: 0.1592183696636372
Fold 3 IBS: 0.18909487165615965
Fold 4 IBS: 0.1617555101933038
Fold 5 IBS: 0.17352826725754153
[I 2024-04-17 14:08:38,613] Trial 80 finished with value: 0.17649331113476227 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 483, 'oob_score': False, 'max_samples': 0.9998847588095947, 'max_features': None, 'min_weight_fraction_leaf': 0.25095162112847785}. Best is trial 66 with value: 0.17608875094401136.
Fold 1 IBS: 0.19645044238241818
Fold 2 IBS: 0.16241201741450556
Fold 3 IBS: 0.18710685872515703
Fold 4 IBS: 0.16169268484725988
Fold 5 IBS: 0.17397072132591115
[I 2024-04-17 14:08:53,881] Trial 81 finished with value: 0.17632654493905037 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 486, 'oob_score': False, 'max_samples': 0.9930359732247087, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.21699983642473847
Fold 2 IBS: 0.1805792093648495
Fold 3 IBS: 0.19889294656533302
Fold 4 IBS: 0.16428998762771174
Fold 5 IBS: 0.1730153178538409
[I 2024-04-17 14:11:49,871] Trial 96 finished with value: 0.18675545956729472 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 9, 'max_depth': 2, 'n_estimators': 455, 'oob_score': False, 'max_samples': 0.8861866003038187, 'max_features': None, 'min_weight_fraction_leaf': 0.1745416008237794}. Best is trial 66 with value: 0.17608875094401136.
Fold 1 IBS: 0.2003194720733052
Fold 2 IBS: 0.19331059374319456
Fold 3 IBS: 0.1850267553757519
Fold 4 IBS: 0.16128969084563635
Fold 5 IBS: 0.1810956383532238
[I 2024-04-17 14:12:02,274] Trial 97 finished with value: 0.18420843007822235 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 3, 'n_estimators': 490, 'oob_score': False, 'max_samples': 0.9073993416752635, 'max_features': None, 'min_weight_fraction_leaf': 0.2

In [52]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [53]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.87
train_ibs:  0.176


#### Test

In [54]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [55]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=14, max_leaf_nodes=20,
                     max_samples=0.8265187125779403, min_samples_leaf=2,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.012312929606053803,
                     n_estimators=195, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.557


RandomSurvivalForest(max_depth=1, max_features=None, max_leaf_nodes=2,
                     max_samples=0.7505526916491491, min_samples_leaf=10,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.1907905531669951,
                     n_estimators=379, oob_score=True, random_state=123)

test_ibs:  0.23


In [56]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [57]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [58]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 14:12:29,593] A new study created in memory with name: no-name-dfcab69a-dd09-4fe7-9c46-363c0fb1cad9


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5367965367965368
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.6764705882352942
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6854460093896714
[I 2024-04-17 14:12:32,412] Trial 0 finished with value: 0.67885353707116 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.67885353707116.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:12:37,628] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. B

Fold 4 C-index: 0.6729957805907173
Fold 5 C-index: 0.6666666666666666
[I 2024-04-17 14:13:34,649] Trial 15 finished with value: 0.6503984303733 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.6897845011357434.
Fold 1 C-index: 0.4675324675324675
Fold 2 C-index: 0.7566964285714286
Fold 3 C-index: 0.6642156862745098
Fold 4 C-index: 0.6666666666666666
Fold 5 C-index: 0.6267605633802817
[I 2024-04-17 14:13:37,565] Trial 16 finished with value: 0.6363743624850707 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is t

Fold 1 C-index: 0.5844155844155844
Fold 2 C-index: 0.7901785714285714
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.7215189873417721
Fold 5 C-index: 0.7699530516431925
[I 2024-04-17 14:14:19,146] Trial 30 finished with value: 0.7271348075932751 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 278, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.05879574792157449}. Best is trial 30 with value: 0.7271348075932751.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7215189873417721
Fold 5 C-index: 0.7699530516431925
[I 2024-04-17 14:14:20,951] Trial 31 finished with value: 0.7200396968216938 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 281, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_

...

Fold 1 IBS: 0.21665779732794135
Fold 2 IBS: 0.21751925717564152
Fold 3 IBS: 0.20412907220401041
Fold 4 IBS: 0.22210633190682874
Fold 5 IBS: 0.2174194451267344
[I 2024-04-17 14:20:08,995] Trial 30 finished with value: 0.21556638074823126 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 16, 'n_estimators': 70, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.7243285885837822, 'min_weight_fraction_leaf': 0.23757381247487194}. Best is trial 26 with value: 0.20425521516318793.
Fold 1 IBS: 0.2122445676054173
Fold 2 IBS: 0.20252464884312424
Fold 3 IBS: 0.19700770961640962
Fold 4 IBS: 0.21086302691479805
Fold 5 IBS: 0.2027758259164211
[I 2024-04-17 14:20:11,284] Trial 31 finished with value: 0.20508315577923408 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 3, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 104, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5

Fold 1 IBS: 0.21630497223947837
Fold 2 IBS: 0.21958595298061412
Fold 3 IBS: 0.2035980172089794
Fold 4 IBS: 0.22215871269338675
Fold 5 IBS: 0.21643846186486954
[I 2024-04-17 14:20:58,930] Trial 45 finished with value: 0.21561722339746564 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 10, 'min_samples_leaf': 6, 'max_depth': 11, 'n_estimators': 83, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.373726821058381, 'min_weight_fraction_leaf': 0.16389807799685063}. Best is trial 32 with value: 0.19086465523101687.
Fold 1 IBS: 0.21673865025266792
Fold 2 IBS: 0.21875173944717735
Fold 3 IBS: 0.20270337232862157
Fold 4 IBS: 0.22216669834462083
Fold 5 IBS: 0.2153324381929384
[I 2024-04-17 14:21:05,573] Trial 46 finished with value: 0.2151385797132052 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 162, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.5931

Fold 1 IBS: 0.21412261619089062
Fold 2 IBS: 0.20096170300523308
Fold 3 IBS: 0.19768334815553634
Fold 4 IBS: 0.20413083791991032
Fold 5 IBS: 0.2016512104175567
[I 2024-04-17 14:21:37,423] Trial 60 finished with value: 0.2037099431378254 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 16, 'n_estimators': 91, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.6341864022990672, 'min_weight_fraction_leaf': 0.04110019733163659}. Best is trial 47 with value: 0.18629015856649767.
Fold 1 IBS: 0.2176747793322257
Fold 2 IBS: 0.18204974561603254
Fold 3 IBS: 0.18642286195176436
Fold 4 IBS: 0.18175695168041173
Fold 5 IBS: 0.17167476077067825
[I 2024-04-17 14:21:39,468] Trial 61 finished with value: 0.18791581987022252 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 42, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.522

Fold 1 IBS: 0.21702643850803088
Fold 2 IBS: 0.17804524130728247
Fold 3 IBS: 0.18851531925648804
Fold 4 IBS: 0.17648057689338645
Fold 5 IBS: 0.17154746817991223
[I 2024-04-17 14:22:15,850] Trial 75 finished with value: 0.18632300882902003 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 20, 'n_estimators': 66, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8255984813979496, 'min_weight_fraction_leaf': 0.040756159127064116}. Best is trial 67 with value: 0.1849635296040363.
Fold 1 IBS: 0.2113961496484921
Fold 2 IBS: 0.1821449835808232
Fold 3 IBS: 0.18903595168291273
Fold 4 IBS: 0.17765728598189454
Fold 5 IBS: 0.17855049104288342
[I 2024-04-17 14:22:18,503] Trial 76 finished with value: 0.18775697238740122 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 93, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.75

Fold 1 IBS: 0.22037856887965046
Fold 2 IBS: 0.18002949433356782
Fold 3 IBS: 0.18406495584411262
Fold 4 IBS: 0.1773755631460241
Fold 5 IBS: 0.1758543766206012
[I 2024-04-17 14:23:11,035] Trial 90 finished with value: 0.18754059176479126 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 15, 'min_samples_leaf': 4, 'max_depth': 20, 'n_estimators': 66, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9979754136756178, 'min_weight_fraction_leaf': 0.06231875931942612}. Best is trial 67 with value: 0.1849635296040363.
Fold 1 IBS: 0.21238270350650948
Fold 2 IBS: 0.18322172515976223
Fold 3 IBS: 0.18710380027472065
Fold 4 IBS: 0.17995136154962357
Fold 5 IBS: 0.1745804020860065
[I 2024-04-17 14:23:14,899] Trial 91 finished with value: 0.18744799851532448 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 18, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 140, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.843

In [59]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.807
train_ibs:  0.184


#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=7, max_features=None, max_leaf_nodes=10,
                   max_samples=0.939348623311968, min_samples_split=5,
                   min_weight_fraction_leaf=0.013770059968835775,
                   n_estimators=322, random_state=123, warm_start=True)

C-index score: 0.58


ExtraSurvivalTrees(max_depth=18, max_features=None, max_leaf_nodes=20,
                   max_samples=0.8754219619832959, min_samples_leaf=2,
                   min_samples_split=15,
                   min_weight_fraction_leaf=0.031831116281223065,
                   n_estimators=259, random_state=123, warm_start=True)

IBS: 0.228


In [63]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [64]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 14:24:03,253] A new study created in memory with name: no-name-39844da6-1e3c-407f-8d9f-cd25da1023b3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:24:42,190] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:25:03,883] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:36:04,757] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.7009449603314964.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:37:25,018] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:51:47,419] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 12 with value: 0.7009449603314964.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:53:33,120] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:18:10,899] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.24849699972703826, 'learning_rate': 0.08739840427890608, 'dropout_rate': 0.9925112981171803, 'n_estimators': 238, 'criterion': 'squared_error', 'ccp_alpha': 1.2571847442990605, 'min_weight_fraction_leaf': 0.13620319457876867, 'max_features': 'auto', 'min_impurity_decrease': 0.00078866474186123, 'validation_fraction': 0.5876510269738553, 'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 42 with value: 0.2140269753656982.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-17 16:18:16,474] Trial 45 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.3318702090173651, 'learning_rate': 0.0758679722545

Fold 3 IBS: 0.20308486363078354
Fold 4 IBS: 0.222059251031902
Fold 5 IBS: 0.21620441591377326
[I 2024-04-17 16:20:06,141] Trial 55 finished with value: 0.214678776568379 and parameters: {'subsample': 0.28519589682406243, 'learning_rate': 0.07016205075701004, 'dropout_rate': 0.97279711727528, 'n_estimators': 121, 'criterion': 'squared_error', 'ccp_alpha': 0.03767053942294234, 'min_weight_fraction_leaf': 0.19525087957715356, 'max_features': None, 'min_impurity_decrease': 0.0005802162560770055, 'validation_fraction': 0.5528235183342366, 'min_samples_split': 3, 'max_leaf_nodes': 15, 'min_samples_leaf': 7, 'max_depth': 9}. Best is trial 42 with value: 0.2140269753656982.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-17 16:20:08,854] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.26773896940389147, 'learning_rate': 0.06921569697149174

Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:20:49,476] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.49624202157362907, 'learning_rate': 0.04172325575959995, 'dropout_rate': 0.871655087593486, 'n_estimators': 150, 'criterion': 'squared_error', 'ccp_alpha': 0.343721568588375, 'min_weight_fraction_leaf': 0.18161893538206386, 'max_features': None, 'min_impurity_decrease': 0.0011338716061792269, 'validation_fraction': 0.2878946538079263, 'min_samples_split': 5, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 13}. Best is trial 42 with value: 0.2140269753656982.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-17 16:21:00,192] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.5856185588208113, 'learning_rate': 0.09086172721039212, 'dropout_rate': 0.8164237696

Fold 5 IBS: 0.21812431525609582
[I 2024-04-17 16:21:41,785] Trial 77 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.5357913465602108, 'learning_rate': 0.09509164545595872, 'dropout_rate': 0.8459672741379245, 'n_estimators': 117, 'criterion': 'squared_error', 'ccp_alpha': 1.0998319625381408, 'min_weight_fraction_leaf': 0.12634401560860808, 'max_features': 0.1, 'min_impurity_decrease': 0.0005815784991320142, 'validation_fraction': 0.7883874326786836, 'min_samples_split': 4, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 15}. Best is trial 42 with value: 0.2140269753656982.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:21:58,234] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.3505708388377867, 'learning_rate': 0.09232934356399627, 'dropout_rate': 0.7064261992556284, 'n_estimators': 241, 'cr

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:22:59,200] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.28396267269938413, 'learning_rate': 0.09422611340750953, 'dropout_rate': 0.9812607710537841, 'n_estimators': 60, 'criterion': 'squared_error', 'ccp_alpha': 5.986001210354111, 'min_weight_fraction_leaf': 0.09373826202704161, 'max_features': None, 'min_impurity_decrease': 0.0024778007971021196, 'validation_fraction': 0.4942203533463335, 'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 15}. Best is trial 42 with value: 0.2140269753656982.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:23:05,530] Trial 90 finished with value: 0.21659054862241586 and parameters: {'subs

In [65]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.701
train_ibs:  0.214


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.058133567268478875,
                                 criterion='squared_error',
                                 dropout_rate=0.2527776031497929,
                                 learning_rate=0.0012227187192111223,
                                 max_depth=1, max_features='auto',
                                 max_leaf_nodes=20,
                                 min_impurity_decrease=1.5094636128058346e-06,
                                 min_samples_leaf=15, min_samples_split=20,
                                 min_weight_fraction_leaf=0.2151989029549357,
                                 n_estimators=489, random_state=123,
                                 subsample=0.873850481285158,
                                 validation_fraction=0.9945176333416724)

C-index score: 0.583


GradientBoostingSurvivalAnalysis(ccp_alpha=0.0025776361420084756,
                                 criterion='squared_error',
                                 dropout_rate=0.7930596249666158,
                                 learning_rate=0.0889199108857109, max_depth=1,
                                 max_features='auto', max_leaf_nodes=13,
                                 min_impurity_decrease=0.0003409042283428851,
                                 min_samples_leaf=11, min_samples_split=17,
                                 min_weight_fraction_leaf=0.13549403283131412,
                                 n_estimators=170, random_state=123,
                                 subsample=0.35070731793187043,
                                 validation_fraction=0.6733131712125798)

IBS: 0.22


In [69]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [70]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [71]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 16:24:53,983] A new study created in memory with name: no-name-21fba87b-a508-427d-ac80-d4432bc785f6


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:24:55,696] Trial 0 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:25:08,884] Trial 1 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6818827456832424.
Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 

Fold 1 C-index: 0.5714285714285714
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.6764705882352942
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.6666666666666666
[I 2024-04-17 16:26:16,045] Trial 19 finished with value: 0.6767159078821401 and parameters: {'subsample': 0.29012679930776175, 'dropout_rate': 0.9852143001156547, 'n_estimators': 431, 'learning_rate': 0.08210359937920125}. Best is trial 3 with value: 0.685290607677292.
Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.696078431372549
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:26:21,310] Trial 20 finished with value: 0.677983096065052 and parameters: {'subsample': 0.44973550553153746, 'dropout_rate': 0.19350037184585095, 'n_estimators': 278, 'learning_rate': 0.06348170552170027}. Best is trial 3 with value: 0.685290607677292.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index:

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:27:25,606] Trial 38 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7039903023611291, 'dropout_rate': 0.14386816981770942, 'n_estimators': 169, 'learning_rate': 0.06713918754706011}. Best is trial 3 with value: 0.685290607677292.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:27:28,916] Trial 39 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7663158695971513, 'dropout_rate': 0.33285705454120956, 'n_estimators': 220, 'learning_rate': 0.01715036845928685}. Best is trial 3 with value: 0.685290607677292.
Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.6764705882352942
Fol

...

Fold 2 IBS: 0.16123008298113736
Fold 3 IBS: 0.1745013121861096
Fold 4 IBS: 0.17691114823310736
Fold 5 IBS: 0.17502712003769116
[I 2024-04-17 16:31:55,357] Trial 19 finished with value: 0.17992669619184318 and parameters: {'subsample': 0.8350722272141606, 'dropout_rate': 0.9887344422073516, 'n_estimators': 235, 'learning_rate': 0.036212385990032875}. Best is trial 2 with value: 0.1763338998360291.
Fold 1 IBS: 0.21495078894573805
Fold 2 IBS: 0.15700881753658336
Fold 3 IBS: 0.17277259067356457
Fold 4 IBS: 0.17353617304957153
Fold 5 IBS: 0.17316931509430378
[I 2024-04-17 16:31:57,584] Trial 20 finished with value: 0.17828753705995226 and parameters: {'subsample': 0.5568571093279632, 'dropout_rate': 0.19350037184585095, 'n_estimators': 161, 'learning_rate': 0.06348170552170027}. Best is trial 2 with value: 0.1763338998360291.
Fold 1 IBS: 0.2324916714901111
Fold 2 IBS: 0.14281050488656433
Fold 3 IBS: 0.17495112113541136
Fold 4 IBS: 0.16606234750928592
Fold 5 IBS: 0.17362532167442254
[I 2024-

Fold 3 IBS: 0.174539339955922
Fold 4 IBS: 0.17677411321719771
Fold 5 IBS: 0.1748171030684222
[I 2024-04-17 16:32:32,335] Trial 38 finished with value: 0.1797801332663459 and parameters: {'subsample': 0.9000660387308665, 'dropout_rate': 0.3720991365773877, 'n_estimators': 92, 'learning_rate': 0.09484943844583128}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.23312185694951
Fold 2 IBS: 0.14152754442207624
Fold 3 IBS: 0.1738623702033595
Fold 4 IBS: 0.16517162749120218
Fold 5 IBS: 0.17320335516077462
[I 2024-04-17 16:32:34,919] Trial 39 finished with value: 0.1773773508453845 and parameters: {'subsample': 0.6998122574699481, 'dropout_rate': 0.47645352742973207, 'n_estimators': 189, 'learning_rate': 0.08850670895881207}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.20655830218659368
Fold 2 IBS: 0.17492142020744073
Fold 3 IBS: 0.17992637231108904
Fold 4 IBS: 0.18738850901139598
Fold 5 IBS: 0.1819660108644699
[I 2024-04-17 16:32:36,517] Trial 40 finished wi

Fold 4 IBS: 0.17323719207666877
Fold 5 IBS: 0.17391114418369052
[I 2024-04-17 16:33:24,462] Trial 57 finished with value: 0.1805083149476045 and parameters: {'subsample': 0.2888110376335854, 'dropout_rate': 0.2571472416667656, 'n_estimators': 198, 'learning_rate': 0.06361079949574666}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.22563954492676833
Fold 2 IBS: 0.1450864819710421
Fold 3 IBS: 0.17178934906165097
Fold 4 IBS: 0.16674907181784904
Fold 5 IBS: 0.17189804627761335
[I 2024-04-17 16:33:26,968] Trial 58 finished with value: 0.17623249881098477 and parameters: {'subsample': 0.9511535444605486, 'dropout_rate': 0.48062274276704253, 'n_estimators': 176, 'learning_rate': 0.0781546863079734}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.2269672945175514
Fold 2 IBS: 0.1439593010574543
Fold 3 IBS: 0.1719657542477086
Fold 4 IBS: 0.16659339591616157
Fold 5 IBS: 0.17212797426047644
[I 2024-04-17 16:33:30,061] Trial 59 finished with value: 0.176322743999870

Fold 5 IBS: 0.1765257577906424
[I 2024-04-17 16:34:08,624] Trial 76 finished with value: 0.18158276527915845 and parameters: {'subsample': 0.5342916507439118, 'dropout_rate': 0.3308474569541405, 'n_estimators': 95, 'learning_rate': 0.08239604675369354}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.22661340611062414
Fold 2 IBS: 0.14452172357008383
Fold 3 IBS: 0.17205945064331313
Fold 4 IBS: 0.16641683545014102
Fold 5 IBS: 0.17205875073324323
[I 2024-04-17 16:34:10,636] Trial 77 finished with value: 0.17633403330148104 and parameters: {'subsample': 0.9609879844906213, 'dropout_rate': 0.5631110327938873, 'n_estimators': 162, 'learning_rate': 0.08696726514452227}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.22066984449071184
Fold 2 IBS: 0.14951880841845128
Fold 3 IBS: 0.17164572765928524
Fold 4 IBS: 0.16909515800883118
Fold 5 IBS: 0.1718206369393307
[I 2024-04-17 16:34:12,649] Trial 78 finished with value: 0.17655003510332207 and parameters: {'subsample

Fold 5 IBS: 0.17342217216926453
[I 2024-04-17 16:34:58,779] Trial 95 finished with value: 0.17830855107952073 and parameters: {'subsample': 0.8589098741489217, 'dropout_rate': 0.598749764600434, 'n_estimators': 173, 'learning_rate': 0.05521411192630663}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.22565089304118113
Fold 2 IBS: 0.1447600949880749
Fold 3 IBS: 0.17360701432529962
Fold 4 IBS: 0.16815975301993338
Fold 5 IBS: 0.1721202735830799
[I 2024-04-17 16:35:01,113] Trial 96 finished with value: 0.17685960579151377 and parameters: {'subsample': 0.5148697353905858, 'dropout_rate': 0.5688549493582022, 'n_estimators': 186, 'learning_rate': 0.07351393012072571}. Best is trial 26 with value: 0.1761905708629028.
Fold 1 IBS: 0.2552690555238246
Fold 2 IBS: 0.14242222735539223
Fold 3 IBS: 0.19832205398740438
Fold 4 IBS: 0.17003726248759515
Fold 5 IBS: 0.18703961727390805
[I 2024-04-17 16:35:07,903] Trial 97 finished with value: 0.19061804332562488 and parameters: {'subsample'

In [72]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [73]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.685
train_ibs:  0.176


#### Test

In [74]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [75]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.15371010694861154,
                                              learning_rate=0.07406154516747154,
                                              n_estimators=200,
                                              random_state=123,
                                              subsample=0.494715020211662)

C-index score: 0.58


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.4868646496386122,
                                              learning_rate=0.09970917701467895,
                                              n_estimators=130,
                                              random_state=123,
                                              subsample=0.6817267275162041)

IBS: 0.228


In [76]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [77]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.870,1.0
ExtraSurvivalTrees,0.807,2.0
CoxElastic,0.737,3.0
CoxLasso,0.724,4.0
CoxPH,0.722,5.0
GradientBoosting,0.701,6.0
ComponentwiseGradientBoosting,0.685,7.0
CoxRidge,0.678,8.0


In [78]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxPH,0.173,2.0
CoxLasso,0.173,2.0
CoxElastic,0.173,2.0
Randomsurvivalforest,0.176,4.5
ComponentwiseGradientBoosting,0.176,4.5
ExtraSurvivalTrees,0.184,6.0
GradientBoosting,0.214,7.0
CoxRidge,0.217,8.0


In [79]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.583,1.0
CoxPH,0.580,3.0
ExtraSurvivalTrees,0.580,3.0
ComponentwiseGradientBoosting,0.580,3.0
CoxLasso,0.579,5.5
CoxElastic,0.579,5.5
CoxRidge,0.563,7.0
Randomsurvivalforest,0.557,8.0


In [80]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
GradientBoosting,0.220,1.0
CoxRidge,0.221,2.0
ExtraSurvivalTrees,0.228,3.5
ComponentwiseGradientBoosting,0.228,3.5
Randomsurvivalforest,0.230,5.0
CoxElastic,0.244,6.0
CoxLasso,0.245,7.0
CoxPH,0.246,8.0


In [81]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/minmax/plsr/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_minmax_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [82]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-17
